# GLIMPSE — Train on Colab (ODL operator)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/swing-research/Glimpse/blob/main/notebooks/glimpse_colab.ipynb)

[**GLIMPSE**](https://github.com/swing-research/Glimpse) ([paper](https://ieeexplore.ieee.org/abstract/document/11018464), *IEEE TMI*) reconstructs a CT image from a **sparse-view sinogram** by predicting **one pixel at a time** from only the sinogram data *local* to that pixel — which gives strong out-of-distribution generalization.

This notebook **reproduces the experiment we run on the cluster**: the exact [`configs/lodopab.yaml`](https://github.com/swing-research/Glimpse/blob/main/configs/lodopab.yaml) setup (128×128, **50 views**, **ODL** `Parallel2dGeometry` + astra_cuda, `circle=False`) trained via `glimpse.engine.train` — the same entry point as `python train.py --config configs/lodopab.yaml`. This is the geometry the published `glimpse.pt` was trained with.

> ⚠️ **A GPU runtime is required** (astra's CUDA backend). **Runtime → Change runtime type → GPU.**

## 1. Get the code

Clone the repository and install the `glimpse` package.

In [ ]:
import os
if not os.path.isdir('/content/Glimpse'):
    !git clone https://github.com/swing-research/Glimpse.git /content/Glimpse
%cd /content/Glimpse
!pip install -q -e .

## 2. Install ODL + astra (GPU forward operator)

The `odl` path uses ODL's `Parallel2dGeometry` with the CUDA `astra_cuda` ray transform. On Colab we install the astra wheel (bundles CUDA) and ODL from source.

> **Important — numpy pin.** Colab's `astra`/`scipy` are built against **numpy 2.0.x**; installing ODL can pull a newer numpy and break their ABI (`cannot import name '_center'`). So we install ODL with `--no-deps` and pin `numpy<2.1`. **If you re-run the install cell, or hit a numpy ABI error, do `Runtime → Restart session` and then run the *import* cell only** (not the install cell again). Tested with `astra-toolbox 2.4.0`, `odl 0.8.2`. (~1–2 min.)

In [ ]:
# astra/scipy on Colab are built for numpy 2.0.x — install ODL with --no-deps
# so it can't upgrade numpy past 2.1, then pin numpy back to be safe.
!pip install -q astra-toolbox
!pip install -q --no-deps 'odl @ git+https://github.com/odlgroup/odl@master'
!pip install -q 'numpy<2.1'
print('\nInstall finished. Now run the import cell below.')
print('If you see a numpy ABI error there: Runtime > Restart session, then run the import cell only.')

In [ ]:
import numpy, torch, astra, odl
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > GPU'
print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('astra', astra.__version__, '| odl', odl.__version__)

## 3. Download the dataset

Small LoDoPaB-CT `train`/`test` subsets and the brain `ood` set (raw images; the ODL operator renders sinograms on the GPU during training). Skipped if already present.

> Also available on the 🤗 Hub: `snapshot_download('AmirEhsan1995/lodopab-ct-glimpse', repo_type='dataset', local_dir='datasets')`.

In [ ]:
import subprocess, zipfile, glob

DOWNLOADS = {
    'datasets/train': 'https://drive.switch.ch/index.php/s/qMlALcE7AZzUPBh/download',
    'datasets/test':  'https://drive.switch.ch/index.php/s/fWBUmtZjozwpN9W/download',
    'datasets/ood':   'https://drive.switch.ch/index.php/s/BQ8Yb8ofjutsEjV/download',
}
os.makedirs('datasets', exist_ok=True)

def ensure_dataset(target_dir, url):
    if os.path.isdir(target_dir) and glob.glob(os.path.join(target_dir, '*')):
        print(f'[ok] {target_dir} already present'); return
    zip_path = target_dir + '.zip'
    print(f'[..] downloading -> {zip_path}')
    subprocess.run(['curl', '-sS', '-L', '-o', zip_path, url], check=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall('datasets')

for target, url in DOWNLOADS.items():
    ensure_dataset(target, url)

print('train:', len(glob.glob('datasets/train/*')),
      '| test:', len(glob.glob('datasets/test/*')),
      '| ood:', len(glob.glob('datasets/ood/*')))

## 4. Configuration (the cluster experiment)

Load the exact cluster config. Everything — geometry, views, patch, learnable filter — is documented inline in [`configs/lodopab.yaml`](https://github.com/swing-research/Glimpse/blob/main/configs/lodopab.yaml).

> `configs/lodopab.yaml` ships with `epochs: 3000` (the full cluster run, several hours on a Colab GPU). For a quick look, lower `config.epochs` below; for the paper-quality checkpoint, leave it.

In [ ]:
import numpy as np
from glimpse import Config

config = Config.from_yaml('configs/lodopab.yaml')
# config.epochs = 300   # <- uncomment for a fast demo instead of the full 3000-epoch run

device = torch.device('cuda')
print('geometry :', config.geometry, '| circle:', config.circle)
print('image_size:', config.image_size, '| n_angles:', config.n_angles, '| epochs:', config.epochs)

## 5. Train

`glimpse.engine.train` runs the full training + periodic evaluation loop — the same code path as `python train.py --config configs/lodopab.yaml` on the cluster. With `geometry='odl'` it builds the ODL operator and generates sinograms on the GPU each step. It writes checkpoints, loss curves, and GT/FBP/GLIMPSE/error figures + PSNR/SSIM to `experiments/128_50_lodopab/`, and returns the model.

Reference numbers from the full 3000-epoch cluster run: **test 38.0 dB / 0.93 SSIM**, **OOD brain 31.6 dB / 0.88** (vs FBP 30.8 / 26.1).

In [ ]:
from glimpse.engine import train

model = train(config)        # trains, evaluates, and writes experiments/128_50_lodopab/
model.eval();

## 6. Visualize reconstructions

Reconstruct a few slices with the trained model and the ODL operator, comparing against the FBP baseline (PSNR / SSIM).

In [ ]:
import matplotlib.pyplot as plt
from glimpse import make_dataset
from glimpse.operators import build_operator
from glimpse.reconstruct import make_coordinate_grid, reconstruct_image
from glimpse.metrics import PSNR, SSIM

true_angles, init_angles = config.resolve_angles()
operator = build_operator(config, np.deg2rad(true_angles))

@torch.no_grad()
def reconstruct_n(directory, n=4):
    ds = make_dataset(directory, config, true_angles, init_angles, network='odl')
    _, volumes = zip(*[ds[i] for i in range(min(n, len(ds)))])
    volume = torch.as_tensor(np.stack(volumes), dtype=torch.float32).to(device)
    gt = volume.cpu().numpy()
    sino = operator.project(volume)
    fbp = operator.fbp(sino).cpu().numpy()
    coords = make_coordinate_grid(config.image_size).unsqueeze(0).expand(len(volumes), -1, -1).to(device)
    recon = reconstruct_image(sino, coords, 1, model, chunk_size=1024)
    recon = recon.reshape(len(volumes), config.image_size, config.image_size)
    return gt, fbp, recon

def show(gt, fbp, recon, title=''):
    n = len(gt)
    fig, ax = plt.subplots(n, 4, figsize=(12, 3 * n)); ax = np.atleast_2d(ax)
    cols = ['FBP', 'GLIMPSE', 'Ground truth', 'Error']
    for i in range(n):
        for j, im in enumerate([fbp[i], recon[i], gt[i], np.abs(gt[i] - recon[i])]):
            ax[i, j].imshow(im, cmap='seismic' if j == 3 else 'gray')
            ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
            if i == 0: ax[i, j].set_title(cols[j])
    if title: fig.suptitle(title, y=1.0, fontsize=14)
    plt.tight_layout(); plt.show()

gt, fbp, recon = reconstruct_n('datasets/test', n=4)
print('TEST  FBP     PSNR %.2f dB | SSIM %.3f' % (PSNR(gt, fbp), SSIM(gt, fbp)))
print('TEST  GLIMPSE PSNR %.2f dB | SSIM %.3f' % (PSNR(gt, recon), SSIM(gt, recon)))
show(gt, fbp, recon, title='In-distribution LoDoPaB-CT test (%d views)' % config.n_angles)

In [ ]:
gt_o, fbp_o, recon_o = reconstruct_n('datasets/ood', n=4)
print('OOD   FBP     PSNR %.2f dB | SSIM %.3f' % (PSNR(gt_o, fbp_o), SSIM(gt_o, fbp_o)))
print('OOD   GLIMPSE PSNR %.2f dB | SSIM %.3f' % (PSNR(gt_o, recon_o), SSIM(gt_o, recon_o)))
show(gt_o, fbp_o, recon_o, title='Out-of-distribution brain images (%d views)' % config.n_angles)

## Next steps

- **Reuse the published weights** instead of training: see [`inference_demo.ipynb`](https://github.com/swing-research/Glimpse/blob/main/notebooks/inference_demo.ipynb) and `GlimpseModel.from_pretrained('AmirEhsan1995/Glimpse')`.
- **Fewer / more views:** change `n_angles` in the config and retrain.
- **Uncalibrated geometry:** [`configs/uncalibrated.yaml`](https://github.com/swing-research/Glimpse/blob/main/configs/uncalibrated.yaml) tells the model jittered angles and recovers them via the learnable sensor geometry.
- **No GPU / astra?** A scikit-image (CPU) operator is available via `configs/skimage.yaml` (train your own model; the published ODL checkpoint won't match that layout).